In [ ]:
%%capture install_log
!pip install -q ultralytics pydicom opencv-python-headless tqdm pandas openpyxl scikit-learn pyyaml

In [ ]:
!pip install monai

In [ ]:
import os
import torch


In [ ]:
# Cihaz Kontrolü
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Aktif Cihaz: {device}")

# T4 x2: kaç GPU var?
n_gpu = torch.cuda.device_count()
print(f"GPU sayısı: {n_gpu}")

# Kaggle Dizin Yolları
DATA_DIR = "/kaggle/input/competitions/rsna-2023-abdominal-trauma-detection"
TRAIN_IMAGES_DIR = os.path.join(DATA_DIR, "train_images")
TRAIN_CSV = os.path.join(DATA_DIR, "train_2024.csv")


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

In [ ]:
df_train = pd.read_csv(TRAIN_CSV)

print(f"Toplam Hasta Sayısı (Train): {df_train['patient_id'].nunique()}")
print(f"Sütun Sayısı: {len(df_train.columns)}")
df_train.head()

**1. Sınıf Dengesizliği (Class Imbalance)**# 

In [ ]:
# Organ bazlı hasar (injury) oranlarını hesaplayalım
injury_cols = ['bowel_injury', 'extravasation_injury', 'kidney_low', 'kidney_high', 
               'liver_low', 'liver_high', 'spleen_low', 'spleen_high']

injury_counts = df_train[injury_cols].sum()

plt.figure(figsize=(12, 6))
sns.barplot(x=injury_counts.values, y=injury_counts.index, palette="Reds_r")
plt.title("Veri Setindeki Yaralanma/Hasar Sınıflarının Dağılımı")
plt.xlabel("Hasta Sayısı")
plt.show()

Keşif: Grafiği çizdiğinizde göreceksiniz ki kidney_high veya bowel_injury vakaları toplam veri setinde oldukça az yer kaplar. Modelin bu sınıfları "görmezden gelmesini" engellemek için daha önce bahsettiğimiz ağırlıklı loss (pos_weight) parametresi hayati önem taşıyacaktır.

Ağırlıklı Önem Oranı: Yarışmanın resmi metriğinde bowel_injury ve extravasation_injury yanlış tahmin edilirse skorunuzu çok kötü etkiler (ceza puanı yüksektir). Analizde bu sınıfların az olduğunu gördüğümüz için, loss fonksiyonuna kesinlikle pozitif ağırlık eklemeliyiz.

2. Çoklu Organ Yaralanmaları (Multi-Label Korelasyonu)

In [ ]:
plt.figure(figsize=(10, 8))
sns.heatmap(df_train[injury_cols].corr(), annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Hasar Türleri Arasındaki İlişki (Korelasyon) Matrisi")
plt.show()

Keşif: Eğer iki hasar türü arasında pozitif bir korelasyon varsa, modelimiz bu organların özelliklerini ortak katmanlarda öğrenirken birbirini destekleyecek şekilde eğtilebilir.

3. Görüntü (Slice) Sayılarındaki Tutarsızlık

In [ ]:
df_meta = pd.read_csv(os.path.join(DATA_DIR, "train_series_meta.csv"))

# Her serinin kaçar görüntü içerdiğini anlamak için (Örnek olarak ilk birkaç seriyi analiz edebiliriz)
print(df_meta.head())

# Hastaların tarama sayıları (Her hastanın birden fazla serisi var mı?)
series_per_patient = df_meta.groupby('patient_id')['series_id'].count()
print(f"\nHasta başına düşen ortalama tarama (seri) sayısı: {series_per_patient.mean():.2f}")